In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/venkat23f1000054/mlp-minilm-model/mlp_model.pt
/kaggle/input/datasets/venkat23f1000054/deberta-v3-base/spm.model
/kaggle/input/datasets/venkat23f1000054/deberta-v3-base/config.json
/kaggle/input/datasets/venkat23f1000054/deberta-v3-base/training_args.bin
/kaggle/input/datasets/venkat23f1000054/deberta-v3-base/tokenizer.json
/kaggle/input/datasets/venkat23f1000054/deberta-v3-base/tokenizer_config.json
/kaggle/input/datasets/venkat23f1000054/deberta-v3-base/model.safetensors
/kaggle/input/datasets/venkat23f1000054/deberta-v3-base/special_tokens_map.json
/kaggle/input/datasets/venkat23f1000054/deberta-v3-base/added_tokens.json
/kaggle/input/datasets/venkat23f1000054/tfidf-baseline-model/tfidf_model.pkl
/kaggle/input/datasets/venkat23f1000054/tfidf-baseline-model/tfidf_vectorizer.pkl
/kaggle/input/datasets/venkat23f1000054/deberta-small/spm.model
/kaggle/input/datasets/venkat23f1000054/deberta-small/config.json
/kaggle/input/datasets/venkat23f1000054/deberta-small/tr

## Imports

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [6]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

Train Shape: (2000, 8)
Test Shape: (500, 7)


In [7]:
!pip install -q transformers accelerate sentencepiece

In [8]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [9]:
MODEL_PATH = "/kaggle/input/datasets/venkat23f1000054/deberta-v3-base"
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH
)
model.to(device)
model.eval()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [10]:
def score_options(prompt, option_texts):

    inputs = tokenizer(
        [prompt] * len(option_texts),
        option_texts,
        truncation=True,
        padding=True,
        max_length=384,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        scores = torch.softmax(outputs.logits, dim=1)[:, 1]
    return scores.cpu().numpy()

In [11]:
options = ["A", "B", "C", "D", "E"]
predictions = []
for _, row in test.iterrows():
    option_texts = [str(row[o]) for o in options]
    scores = score_options(
        row["prompt"],
        option_texts
    )
    ranked = np.argsort(scores)[::-1]
    top3 = [options[i] for i in ranked[:3]]
    predictions.append(" ".join(top3))

In [12]:
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [13]:
sample["Prediction"] = predictions
sample.head()

,ID,Prediction
0,1,A E D
1,2,B C A
2,3,B E D
3,4,E C A
4,5,C B D


In [14]:
sample.to_csv("submission1.csv", index=False)